# Flujo de trabajo optimizado

A manera de ejemplo, se va a mostrar como se implementaría un flujo de trabajo optimizado para entrenar un modelo de ML, usando los métodos `Pipeline` y `ColumnTransformer`. Este flujo tendría los siguientes pasos:

- Partición de los datos en subconjuntos de entrenamiento y prueba.
- Creación del bloque de procesamiento con `ColumnTransformer`.
- Creación del flujo integrado de procesamiento y modelado con `Pipeline`.
- Entrenamiento del modelo, y evaluación del mismo.
- Uso del modelo entrenado para hacer predicciones.

## Ejemplo de uso

Se va a entrenar un modelo con el dataset **adult.data**, que tiene como objetivo predecir la variable **income**.

Sa va a hacer el siguiente preprocesamiento a las variables del dataset:

* **age** y **hours-per-week** se van a estandarizar con `StandardScaler`.
* **fnlwgt** se va a transformar con `PowerTransformer`.
* **education** se va a codificar con `OrdinalEncoder`.
* **marital-status**, **occupation** y **sex** se codificarán mediante `OneHotEncoder`.
* Las otras variables se van a descartar del modelo.

Empezamos cargando el dataset, y por facilidad, eliminando los datos nulos y duplicados.

In [1]:
import pandas as pd

df = pd.read_csv(
    "http://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    header =None,
    names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
             'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
             'hours-per-week', 'native-country', 'income'],
    na_values= [' ?']
    )

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Paso 1: Partición de los datos

A continuación, partimos los datos en subconjuntos de entrenamiento y prueba.En este paso, no es necesario descartar las variables que no se van a usar para entrenar el modelo, ya que esto se puede hacer luego en el paso 2.

In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(
    ['income'],
    axis=1
    )

y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, train_size=0.8)

print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

Tamaño del conjunto de entrenamiento es: (24111, 14)
Tamaño del conjunto de prueba es: (6028, 14)


In [3]:
X_train

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
26312,28,Private,293398,HS-grad,9,Separated,Sales,Unmarried,Black,Female,0,0,40,United-States
23791,34,Private,293900,11th,7,Married-spouse-absent,Craft-repair,Not-in-family,Black,Male,0,0,55,United-States
14775,59,Private,233312,Some-college,10,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States
14866,59,Private,159724,Masters,14,Married-civ-spouse,Sales,Husband,White,Male,7298,0,55,United-States
25476,51,Local-gov,96190,Some-college,10,Married-civ-spouse,Adm-clerical,Wife,White,Female,0,0,40,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17289,27,Private,120155,HS-grad,9,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,39,United-States
5192,33,Private,192644,HS-grad,9,Separated,Handlers-cleaners,Unmarried,White,Male,0,0,35,Puerto-Rico
12172,30,Private,189759,Bachelors,13,Never-married,Transport-moving,Not-in-family,White,Male,4865,0,40,United-States
235,42,Self-emp-not-inc,303044,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,Asian-Pac-Islander,Male,0,0,40,Cambodia


### Paso 2: Creación del bloque de procesamiento:

Empezamos creando los transformadores que vamos a utilizar.

In [5]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, PowerTransformer

ss = StandardScaler() # Para preprocesar age y hours-per-week
pt = PowerTransformer() # Para preprocesar fnlwgt
orden = [' Preschool', ' 1st-4th', ' 5th-6th', ' 7th-8th', ' 9th', ' 10th', ' 11th', ' 12th',
         ' HS-grad',' Some-college', ' Prof-school', ' Assoc-acdm', ' Assoc-voc', ' Bachelors',
         ' Masters', ' Doctorate']
ore = OrdinalEncoder(categories=[orden], dtype='int') # Para preprocesar education
ohe = OneHotEncoder(sparse_output=False, drop='if_binary') # marital-status, occupation y sex

Luego creamos el bloque de procesamiento con `ColumnTransformer`:

In [7]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('age_hours', ss, ['age', 'hours-per-week']),
    ('fnlwgt', pt, ['fnlwgt']),
    ('education', ore, ['education']),
    ('marital_occupation_sex', ohe, ['marital-status', 'occupation', 'sex'])
    ],
    remainder='drop') # El resto de las características se descartan

preprocessor

ColumnTransformer(transformers=[('age_hours', StandardScaler(),
                                 ['age', 'hours-per-week']),
                                ('fnlwgt', PowerTransformer(), ['fnlwgt']),
                                ('education',
                                 OrdinalEncoder(categories=[[' Preschool',
                                                             ' 1st-4th',
                                                             ' 5th-6th',
                                                             ' 7th-8th', ' 9th',
                                                             ' 10th', ' 11th',
                                                             ' 12th',
                                                             ' HS-grad',
                                                             ' Some-college',
                                                             ' Prof-school',
                                                             ' Assoc-acdm',
                                                             ' Assoc-voc',
                                                             ' Bachelors',
                                                             ' Masters',
                                                             ' Doctorate']],
                                                dtype='int'),
                                 ['education']),
                                ('marital_occupation_sex',
                                 OneHotEncoder(drop='if_binary',
                                               sparse_output=False),
                                 ['marital-status', 'occupation', 'sex'])])

Luego, entrenamos los transformadores con los datos de entrenamiento:

### Paso 3: Creación del flujo de trabajo integrado:

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=10000))
    ])

pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('age_hours', StandardScaler(),
                                                  ['age', 'hours-per-week']),
                                                 ('fnlwgt', PowerTransformer(),
                                                  ['fnlwgt']),
                                                 ('education',
                                                  OrdinalEncoder(categories=[[' '
                                                                              'Preschool',
                                                                              ' '
                                                                              '1st-4th',
                                                                              ' '
                                                                              '5th-6th',
                                                                              ' '
                                                                              '7th-8th',
                                                                              ' '
                                                                              '9th',
                                                                              ' '
                                                                              '10th',
                                                                              ' '
                                                                              '11th',
                                                                              ' '
                                                                              '12th',
                                                                              ' '
                                                                              'HS-grad',
                                                                              ' '
                                                                              'Some-college',
                                                                              ' '
                                                                              'Prof-school',
                                                                              ' '
                                                                              'Assoc-acdm',
                                                                              ' '
                                                                              'Assoc-voc',
                                                                              ' '
                                                                              'Bachelors',
                                                                              ' '
                                                                              'Masters',
                                                                              ' '
                                                                              'Doctorate']],
                                                                 dtype='int'),
                                                  ['education']),
                                                 ('marital_occupation_sex',
                                                  OneHotEncoder(drop='if_binary',
                                                                sparse_output=False),
                                                  ['marital-status',
                                                   'occupation', 'sex'])])),
                ('model', LogisticRegression(max_iter=10000))])

### Paso 4: Entrenamiento del modelo

Vamos a entrenar un modelo lineal de regresión logística:

In [10]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('age_hours', StandardScaler(),
                                                  ['age', 'hours-per-week']),
                                                 ('fnlwgt', PowerTransformer(),
                                                  ['fnlwgt']),
                                                 ('education',
                                                  OrdinalEncoder(categories=[[' '
                                                                              'Preschool',
                                                                              ' '
                                                                              '1st-4th',
                                                                              ' '
                                                                              '5th-6th',
                                                                              ' '
                                                                              '7th-8th',
                                                                              ' '
                                                                              '9th',
                                                                              ' '
                                                                              '10th',
                                                                              ' '
                                                                              '11th',
                                                                              ' '
                                                                              '12th',
                                                                              ' '
                                                                              'HS-grad',
                                                                              ' '
                                                                              'Some-college',
                                                                              ' '
                                                                              'Prof-school',
                                                                              ' '
                                                                              'Assoc-acdm',
                                                                              ' '
                                                                              'Assoc-voc',
                                                                              ' '
                                                                              'Bachelors',
                                                                              ' '
                                                                              'Masters',
                                                                              ' '
                                                                              'Doctorate']],
                                                                 dtype='int'),
                                                  ['education']),
                                                 ('marital_occupation_sex',
                                                  OneHotEncoder(drop='if_binary',
                                                                sparse_output=False),
                                                  ['marital-status',
                                                   'occupation', 'sex'])])),
                ('model', LogisticRegression(max_iter=10000))])

Y lo evaluamos:

In [11]:
print(f'Exactitud del modelo en el conjunto de entrenamiento: {pipeline.score(X_train, y_train)}')
print(f'Exactitud del modelo en el conjunto de prueba: {pipeline.score(X_test, y_test)}')

Exactitud del modelo en el conjunto de entrenamiento: 0.826427771556551
Exactitud del modelo en el conjunto de prueba: 0.8173523556735236


### Paso 5: Usar el modelo para hacer predicciones:

Supongamos que queremos hacer una predicción con los siguientes datos:
- age: 50
- hours-per-week: 45
- fnlwgt: 100000
- education: Masters
- marital-status: Married-civ-spouse
- occupation: Sales
- sex: Male

Primero, deberíamos crear un dataframe con estos datos y luego hacer la predicción:

In [12]:
data_new = [50, 45, 100000, ' Masters', ' Married-civ-spouse', ' Sales', ' Male']
data_new = pd.DataFrame(data_new).T
data_new.columns = ['age', 'hours-per-week', 'fnlwgt', 'education', 'marital-status', 'occupation', 'sex']
data_new

,age,hours-per-week,fnlwgt,education,marital-status,occupation,sex
0,50,45,100000,Masters,Married-civ-spouse,Sales,Male


In [14]:
y_new = pipeline.predict(data_new)
y_new

array([' >50K'], dtype=object)